# 04 — Portföy simülasyonu ve araştırma sonuçları

Bu defter 2024 tahminlerinden ters oynaklık ağırlıkları oluşturur ve eşit ağırlıkla maliyet duyarlılığı altında karşılaştırır. Bu, kaldıraçsız endeks tabanlı bir araştırma simülasyonudur.

In [1]:
from pathlib import Path
import json, tomllib
import pandas as pd
import plotly.express as px
from IPython.display import display
ROOT = Path.cwd() if (Path.cwd() / "config.toml").exists() else Path.cwd().parent
RAW, OUT = ROOT / "data/private/study-yahoo-real", ROOT / "results/research"
assert RAW.exists(), "Gerçek ham girdi data/private altında hazırlanmalıdır."
config = tomllib.loads((ROOT / "config.toml").read_text(encoding="utf-8"))
START, END, SECTORS = pd.Timestamp(config["study"]["start"]), pd.Timestamp(config["study"]["end"]), config["study"]["sectors"]
print(f"Çalışma dönemi: {START.date()} — {END.date()}")
print("Temsilciler:", ", ".join(SECTORS))

Çalışma dönemi: 2019-01-01 — 2024-12-31
Temsilciler: XBANK, XUSIN


In [2]:
predictions=pd.read_csv(OUT/"predictions.csv",parse_dates=["origin","target_date","fit_until"])
metrics_table=pd.read_csv(OUT/"metrics.csv"); quality=pd.read_csv(OUT/"quality.csv")
assert (predictions.target_date.dt.year == config["study"]["test_year"]).all(); display(metrics_table)

,sector,target,model,n,mae,rmse,mae_low,mae_high,baseline_mae,mae_improvement
0,XBANK,return,historical_mean,12,0.079159,0.100611,0.053450,0.106123,0.079159,0.000000
1,XBANK,return,persistence,12,0.112363,0.140772,0.065689,0.173808,0.079159,-0.419457
2,XBANK,return,ridge,12,0.076922,0.096858,0.050294,0.102955,0.079159,0.028268
3,XBANK,return,xgboost,12,0.072018,0.090658,0.046337,0.096769,0.079159,0.090209
4,XBANK,volatility,historical_mean,12,0.051355,0.061330,0.033805,0.074311,0.051355,0.000000
5,XBANK,volatility,persistence,12,0.076692,0.100177,0.045769,0.109533,0.051355,-0.493362
6,XBANK,volatility,ridge,12,0.067291,0.077527,0.047948,0.091090,0.051355,-0.310301
7,XBANK,volatility,xgboost,12,0.058866,0.068057,0.039154,0.084410,0.051355,-0.146255
8,XUSIN,return,historical_mean,12,0.059452,0.068857,0.043494,0.079491,0.059452,0.000000
9,XUSIN,return,persistence,12,0.073001,0.095483,0.047339,0.104297,0.059452,-0.227911


In [3]:
from bist_risk.portfolio import simulate
portfolio,portfolio_summary,weights=simulate(predictions,config["portfolio"]["cost_bps"],config["portfolio"]["vol_floor"])
import numpy as np
np.testing.assert_allclose(weights.groupby(["date","strategy"]).weight.sum(),1)
display(portfolio_summary); display(weights.head())

,strategy,cost_bps,total_return,max_drawdown,months
0,inverse_forecast_vol,0,0.343867,-0.170687,12
1,inverse_forecast_vol,10,0.341287,-0.170993,12
2,inverse_forecast_vol,25,0.337422,-0.171453,12
3,inverse_forecast_vol,50,0.330999,-0.172218,12
4,equal_weight,0,0.391718,-0.178243,12
5,equal_weight,10,0.389716,-0.178307,12
6,equal_weight,25,0.386717,-0.178403,12
7,equal_weight,50,0.381726,-0.178563,12


,date,strategy,sector,weight
0,2024-01-31,inverse_forecast_vol,XBANK,0.424478
1,2024-01-31,inverse_forecast_vol,XUSIN,0.575522
2,2024-02-29,inverse_forecast_vol,XBANK,0.396539
3,2024-02-29,inverse_forecast_vol,XUSIN,0.603461
4,2024-03-31,inverse_forecast_vol,XBANK,0.396543


In [4]:
px.line(portfolio[portfolio.cost_bps.eq(10)],x="date",y="wealth",color="strategy",title="Gerçek veri — 10 bps maliyet altında portföy değeri").show()
px.bar(portfolio_summary,x="cost_bps",y="total_return",color="strategy",barmode="group",title="Maliyet duyarlılığı: toplam getiri").show()

## Sonuçların sürümlenmesi

Bu hücre, önceki üç notebook’un ürettiği tabloları hash’ler; kaynak, ayar ve paket bilgilerini manifestte kaydeder. Ham dosyalar manifestte yalnızca hash olarak temsil edilir.

In [5]:
from bist_risk.artifacts import write_tables,create_manifest,write_turkish_report
write_tables(OUT,{"portfolio":portfolio,"portfolio_summary":portfolio_summary,"weights":weights})
names=["monthly","macro","shocks","quality","adf","causality","predictions","model_selection","metrics","portfolio","portfolio_summary","weights"]
tables={name:pd.read_csv(OUT/f"{name}.csv") for name in names}
manifest=create_manifest(OUT,config,json.loads((RAW/"provenance.json").read_text(encoding="utf-8")),SECTORS,RAW,tables)
write_turkish_report(OUT,manifest); print("Manifest ve Türkçe rapor gerçek notebook sonuçlarından üretildi.")

Manifest ve Türkçe rapor gerçek notebook sonuçlarından üretildi.


In [6]:
from bist_risk.artifacts import sha256
for name,digest in manifest["result_hashes"].items(): assert sha256(OUT/f"{name}.csv")==digest
print("Tüm sonuç hashleri doğrulandı.")

Tüm sonuç hashleri doğrulandı.
